# XE4 GEMM subgroup-split reduce

A **full-execution-flow** functional port of the sycl-tla example [`examples/cute/tutorial/xe4/gemm_sgsplit_reduce_xe4.cpp`](../../examples/cute/tutorial/xe4/gemm_sgsplit_reduce_xe4.cpp ), runnable on the CPU.

K split across subgroups, then reduced.

The cells reproduce the real kernel flow — setup → config → tiling → SLM staging → clear accumulator → prologue → **K-loop mainloop** → epilogue → store → **validation**. Each cell does one operation and uses the shared display helpers from the first code cell. (Same content as `gemm_sgsplit_reduce_xe4.py`.)

## Helpers — display + SLM copy (reused by every cell below)

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True, linewidth=120)
from tensor_layouts import Layout, size
from tensor_layouts.analysis import is_bijective
from tensor_layouts.atoms_xe_common import make_slm_layout_elem, sizeof_bits

# ---- display helpers (reused by every cell below) ----
def show_mat(name, X, r=4, c=8):
    X = np.asarray(X)
    print(name, " shape", X.shape)
    print(X[:r, :c] if X.ndim == 2 else X[:c])
    if X.ndim == 2 and (X.shape[0] > r or X.shape[1] > c):
        print("   ...(showing %dx%d of %dx%d)" % (min(r, X.shape[0]), min(c, X.shape[1]), *X.shape))

def show_layout(layout, n_rows, n_cols, rl="m", cl="k", max_r=8, max_c=8):
    R, C = min(n_rows, max_r), min(n_cols, max_c)
    print("      " + "".join((cl + str(j)).ljust(5) for j in range(C)) + (" ..." if C < n_cols else ""))
    for i in range(R):
        print((" " + rl + str(i)).ljust(6) + "".join(str(layout(i, j)).ljust(5) for j in range(C))
              + (" ..." if C < n_cols else ""))
    if R < n_rows:
        print("   ...(%dx%d total)" % (n_rows, n_cols))

def check(name, cond):
    print(("[PASS] " if cond else "[FAIL] ") + name)

# ---- SLM staging helpers: gmem tile <-> bank-swizzled SLM buffer ----
def load_to_slm(mat, lay):
    buf = np.zeros(size(lay), dtype=mat.dtype)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            buf[lay(i, j)] = mat[i, j]
    return buf

def read_slm(buf, lay, r, c):
    return np.array([[buf[lay(i, j)] for j in range(c)] for i in range(r)])

print("helpers ready: show_mat, show_layout, check, load_to_slm, read_slm")

helpers ready: show_mat, show_layout, check, load_to_slm, read_slm


## Setup — allocate + initialize operands

In [2]:
# Problem setup: allocate and initialize the operands (small so tables print).
M, N, K = 32, 32, 64
BLK_M, BLK_N, BLK_K = 32, 32, 16
K_TILE = K // BLK_K            # number of K-tiles in the mainloop
K_PIPE = 2                    # SLM pipeline stages (StagesA)
alpha, beta = 1.0, 0.0
DT = "bf16"; DBITS = sizeof_bits(DT)
rng = np.random.default_rng(0)
A = rng.integers(-2, 3, size=(M, K)).astype(np.float32)   # A (M x K) row-major
B = rng.integers(-2, 3, size=(N, K)).astype(np.float32)   # B (N x K); B^T used in the GEMM
C = rng.integers(0, 2, size=(M, N)).astype(np.float32)    # C (M x N) input accumulator
show_mat("A (M x K)", A); show_mat("B (N x K)", B)

A (M x K)  shape (32, 64)
[[ 2.  1.  0. -1. -1. -2. -2. -2.]
 [ 2. -2.  0.  1.  2.  0. -1. -1.]
 [ 2. -1.  2.  1.  2. -2.  1.  2.]
 [-1. -2.  0.  2. -1.  2. -1.  2.]]
   ...(showing 4x8 of 32x64)
B (N x K)  shape (32, 64)
[[ 2.  0.  0. -1.  2. -2.  2. -2.]
 [ 1. -1. -2.  2.  1. -2.  0.  1.]
 [ 2.  0. -1.  2.  2. -1.  2.  1.]
 [ 2.  0.  2.  0.  1. -1. -2.  1.]]
   ...(showing 4x8 of 32x64)


## Config — tile shape, MMA atom, SLM layouts, pipeline stages

In [3]:
from tensor_layouts.atoms_xe4 import make_xe4_amma_atom
atom = make_xe4_amma_atom("f32", DT, DT, "f32", BLK_M, BLK_N, BLK_K, tracking="AB")
slmA = make_slm_layout_elem(DBITS, BLK_M, BLK_K)
slmB = make_slm_layout_elem(DBITS, BLK_N, BLK_K)
print("MMA atom :", atom.name)
print("CTA tile :", (BLK_M, BLK_N, BLK_K), " K-tiles:", K_TILE, " pipeline stages:", K_PIPE)

MMA atom : XE4_AMMA_AB_32x32x16_F32BF16BF16F32
CTA tile : (32, 32, 16)  K-tiles: 4  pipeline stages: 2


## Tiling — CTA grid + K-tiles

In [4]:
# Partition the problem: CTA grid over (M,N), and K split into K_TILE tiles.
print("CTA grid (M,N):", (M // BLK_M, N // BLK_N), "   K-tiles per CTA:", K_TILE)
a_kt = [A[:, k * BLK_K:(k + 1) * BLK_K] for k in range(K_TILE)]   # A K-tiles (M x BLK_K)
b_kt = [B[:, k * BLK_K:(k + 1) * BLK_K] for k in range(K_TILE)]   # B K-tiles (N x BLK_K)
show_mat("A K-tile 0 (M x BLK_K)", a_kt[0])

CTA grid (M,N): (1, 1)    K-tiles per CTA: 4
A K-tile 0 (M x BLK_K)  shape (32, 16)
[[ 2.  1.  0. -1. -1. -2. -2. -2.]
 [ 2. -2.  0.  1.  2.  0. -1. -1.]
 [ 2. -1.  2.  1.  2. -2.  1.  2.]
 [-1. -2.  0.  2. -1.  2. -1.  2.]]
   ...(showing 4x8 of 32x16)


## SLM staging — bank-swizzled operand tiles

In [5]:
# SLM staging: each K-tile is copied into a bank-swizzled SLM buffer.
print("SLM A stage layout (m,k) -> SLM offset (bank-swizzled):")
show_layout(slmA, BLK_M, BLK_K)
_probe = load_to_slm(a_kt[0], slmA)
check("SLM stage round-trip is loss-less", np.array_equal(read_slm(_probe, slmA, BLK_M, BLK_K), a_kt[0]))

SLM A stage layout (m,k) -> SLM offset (bank-swizzled):
      k0   k1   k2   k3   k4   k5   k6   k7    ...
 m0   0    1    2    3    4    5    6    7     ...
 m1   16   17   18   19   20   21   22   23    ...
 m2   128  129  130  131  132  133  134  135   ...
 m3   144  145  146  147  148  149  150  151   ...
 m4   256  257  258  259  260  261  262  263   ...
 m5   272  273  274  275  276  277  278  279   ...
 m6   384  385  386  387  388  389  390  391   ...
 m7   400  401  402  403  404  405  406  407   ...
   ...(32x16 total)
[PASS] SLM stage round-trip is loss-less


## Clear the accumulator

In [6]:
# Clear the accumulator fragment (lives in registers on hardware).
acc = np.zeros((BLK_M, BLK_N), dtype=np.float32)
show_mat("cleared accumulator", acc)

cleared accumulator  shape (32, 32)
[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]
   ...(showing 4x8 of 32x32)


## Prologue — fill the SLM pipeline stages

In [7]:
# Prologue: fill the first K_PIPE SLM pipeline stages (double-buffering).
slm_pipe_a = [load_to_slm(a_kt[p], slmA) for p in range(min(K_PIPE, K_TILE))]
slm_pipe_b = [load_to_slm(b_kt[p], slmB) for p in range(min(K_PIPE, K_TILE))]
print("prologue filled", len(slm_pipe_a), "of", K_PIPE, "SLM stages;", K_TILE, "K-tiles total")

prologue filled 2 of 2 SLM stages; 4 K-tiles total


## Mainloop — K-tile load + MMA accumulation

In [8]:
# Split-K mainloop: two workers each accumulate half of the K-tiles; their
# partials are combined by an atomic store-reduce.
mid = K_TILE // 2
acc0 = np.zeros((BLK_M, BLK_N), np.float32)
acc1 = np.zeros((BLK_M, BLK_N), np.float32)
for kt in range(0, mid):
    acc0 += a_kt[kt] @ b_kt[kt].T
for kt in range(mid, K_TILE):
    acc1 += a_kt[kt] @ b_kt[kt].T
show_mat("worker-0 partial", acc0)
show_mat("worker-1 partial", acc1)
acc = acc0 + acc1                                    # store-reduce (atomic add)
show_mat("reduced acc", acc)
check("split-K reduce == A.B^T", np.allclose(acc, A @ B.T))

worker-0 partial  shape (32, 32)
[[  2.  -3.  -7.  24.  13.  32. -12.   2.]
 [  5.   9.   6.   0.  13. -18.   9.  -6.]
 [ 12.  21.   5.  26.  -9. -12.  -4.  13.]
 [-16.   5.  11.   2. -24.  -8.   5.  -8.]]
   ...(showing 4x8 of 32x32)
worker-1 partial  shape (32, 32)
[[ -5. -11.  13.  16.  -7.  -1.   5.  -6.]
 [  2. -28.   4.  -9.  10.   3.  22.   0.]
 [  6.  13. -18.  10.  -9.  -6. -15.   0.]
 [ -2.   3.   2.   7.   2.   2.  15.   7.]]
   ...(showing 4x8 of 32x32)
reduced acc  shape (32, 32)
[[ -3. -14.   6.  40.   6.  31.  -7.  -4.]
 [  7. -19.  10.  -9.  23. -15.  31.  -6.]
 [ 18.  34. -13.  36. -18. -18. -19.  13.]
 [-18.   8.  13.   9. -22.  -6.  20.  -1.]]
   ...(showing 4x8 of 32x32)
[PASS] split-K reduce == A.B^T


## Epilogue

In [9]:
# Epilogue: D = alpha*acc + beta*C.
D = alpha * acc + beta * C
show_mat("D = alpha*acc + beta*C", D)

D = alpha*acc + beta*C  shape (32, 32)
[[ -3. -14.   6.  40.   6.  31.  -7.  -4.]
 [  7. -19.  10.  -9.  23. -15.  31.  -6.]
 [ 18.  34. -13.  36. -18. -18. -19.  13.]
 [-18.   8.  13.   9. -22.  -6.  20.  -1.]]
   ...(showing 4x8 of 32x32)


## Store → gmem

In [10]:
# Store D to gmem through the SLM core matrix; verify loss-less.
slmD = make_slm_layout_elem(DBITS, M, N)
dbuf = load_to_slm(D, slmD)
check("store round-trip", np.array_equal(read_slm(dbuf, slmD, M, N), D))

[PASS] store round-trip


## Reference + validation

In [11]:
# Reference GEMM + validation (like validate_gemm_result).
D_ref = alpha * (A @ B.T) + beta * C
err = np.abs(D - D_ref).max()
print("max abs error vs reference:", err)
check("GEMM verification", err < 1e-3)

max abs error vs reference: 0.0
[PASS] GEMM verification


## Recap

load → tile → stage → clear → prologue → K-loop MMA → epilogue → store → validate.